In [1]:
from __future__ import annotations

import copy
import csv
import json
import os
import time
from dataclasses import dataclass
from typing import Dict, Iterable, List, Optional, Tuple
from loguru import logger
import torch
from torch import nn
from data.common import ProblemClass
from torch.optim import Adam
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
# Your codebase imports (run from src/)
from pathlib import Path
from jobs.utils import build_dataloaders, set_all_seeds, device_from_args, pack_by_sizes
from jobs.finetune import eval_epoch
from metrics.optimization import get_optimal_solution, get_optimality_gap, kkt
from models.gnn import GNNPolicy
from models.mlp import KKTNetMLP
import configargparse

In [2]:
out_dir = Path("./results/budget_sweep")
out_dir.mkdir(parents=True, exist_ok=True)
parser = configargparse.ArgumentParser(
    allow_abbrev=False,
    default_config_files=["../configs/finetune/finetune_CA_200/finetune_CA_200_gnn_full_finetune.yml"],
)
parser.add_argument(
        "--encoder_path",
        type=str,
        help="Path to encoder-only checkpoint from pretraining (e.g., .../best_encoder.pt).",
    )
parser.add_argument("--devices", type=str, default="0")
parser.add_argument("--batch_size", type=int, default=8)
parser.add_argument("--num_workers", type=int, default=0)
parser.add_argument("--seed", type=int, default=42)
parser.add_argument(
        "--finetune_mode",
        type=str,
        choices=["full", "heads"],
        default="full",
        help="'full' updates encoder+heads; 'heads' freezes encoder and trains heads only.",
    )

d = parser.add_argument_group("data")

d.add_argument(
    "--use_bipartite_graphs", action="store_true", help="Must be set for GNNPolicy."
)
d.add_argument(
    "--problems",
    type=str,
    nargs="+",
    default=[ProblemClass.COMBINATORIAL_AUCTION],
    help="Problem type(s).",
)
d.add_argument(
    "--is_sizes", type=int, nargs="+", default=[200]
)
d.add_argument(
    "--ca_sizes", type=int, nargs="+", default=[100]
)
d.add_argument(
    "--sc_sizes", type=int, nargs="+", default=[200]
)
d.add_argument(
    "--cfl_sizes", type=int, nargs="+", default=[200]
)
d.add_argument(
    "--rnd_sizes", type=int, nargs="+", default=[200]
)

d.add_argument("--n_instances", type=int, default=35000)
d.add_argument("--data_root", type=str, default="../data/instances")
d.add_argument(
    "--val_split",
    type=float,
    default=0.15,
    help="Validation split ratio (default: 0.15 for 70/15/15 train/val/test)",
)
args, _ = parser.parse_known_args()
GNNPolicy.add_args(parser)
KKTNetMLP.add_args(parser)

args, _ = parser.parse_known_args()

In [3]:
print("Arguments:", args)

Arguments: Namespace(encoder_path='/home/joachim-verschelde/Repos/KKT_MPNN/src/experiments/kkt_gnn_node_pretraining_experiments/run_gnn_policy-gv=2-lv=8-slice=512-point=13-lmbd=0.256-std=0.0-dim=128-l_mask=0.6-g_mask=0.05-l_e_mask=0.05-g_e_mask=0.6-dp=0.05-l_dim=128-conv=gatv2-heads=4_20260117_222826/best_encoder.pt', devices='0', batch_size=80, num_workers=0, seed=42, finetune_mode='full', use_bipartite_graphs=True, problems=['CA'], is_sizes=[200], ca_sizes=[100], sc_sizes=[200], cfl_sizes=[200], rnd_sizes=[200], n_instances=50000, data_root='../data/instances', val_split=0.15, lejepa_n_global_views=2, lejepa_n_local_views=8, sigreg_slices=512, sigreg_points=13, lejepa_lambda=0.256, lejepa_std_loss_weight=0.0, embedding_size=128, cons_nfeats=4, edge_nfeats=1, var_nfeats=18, num_emb_type='periodic', num_emb_bins=32, num_emb_freqs=16, lejepa_local_mask=0.6, lejepa_global_mask=0.05, lejepa_local_edge_mask=0.05, lejepa_global_edge_mask=0.6, dropout=0.05, lejepa_embed_dim=128, bipartite_co

In [4]:
set_all_seeds(args.seed)
device = device_from_args(args)

In [5]:
resolved_data_root = os.path.abspath(os.path.expanduser(args.data_root))
print(f"Using data root: {resolved_data_root}")

Using data root: /home/joachim-verschelde/Repos/KKT_MPNN/src/data/instances


In [6]:
def _get_state_dict(pkg):
    if isinstance(pkg, dict):
        if "model" in pkg:
            return pkg["model"]
        if "state_dict" in pkg:
            return pkg["state_dict"]
    return pkg  # assume raw state_dict

def infer_MN_from_ckpt(ckpt_path: Path):
    
    pkg = torch.load(ckpt_path, map_location="cpu")
    sd = _get_state_dict(pkg)

    Din = sd["encoder.trunk.0.weight"].shape[1]

    # adjust keys if your MLP uses different names
    M = sd["head_lam.2.bias"].shape[0]
    N = sd["head_x.2.bias"].shape[0]  # <-- if your x-head is named differently, update this

    expected = M * N + M + N
    if Din != expected:
        raise ValueError(f"Inconsistent dims: Din={Din}, but M*N+M+N={expected} (M={M}, N={N})")
    print(f"M:{M} N:{N}")
    return M, N


In [7]:
experiment_dir = Path("/home/joachim-verschelde/Repos/KKT_MPNN/src/experiments/kkt_gnn_node_finetuning_experiments/")
sizes = [10,50,200]
datasets =["ca","all"]
scenarios=["mlp","gnn","gnnff","gnnfe"]
checkpoint_epochs = [1,2,5,10,20,30,50]
out_csv = experiment_dir / "exp1_budget_sweep_results.csv"

# Cache dataloaders per (dataset,size,use_bipartite)
dl_cache: Dict[Tuple[str, int, bool], Tuple] = {}

rows: List[Dict[str, object]] = []

for size in tqdm(sizes):
    size_dir = experiment_dir / Path(str(size))
    for dataset in tqdm(datasets):
        for scenario in tqdm(scenarios):
            model_dir = list(size_dir.glob(f"*-{dataset}-{scenario}"))[0]
            args.problems = [ProblemClass.COMBINATORIAL_AUCTION] if dataset == "ca" else [ProblemClass.CAPACITATED_FACILITY_LOCATION, ProblemClass.SET_COVER, ProblemClass.INDEPENDANT_SET, ProblemClass.COMBINATORIAL_AUCTION]

            args.is_sizes = [size]
            args.ca_sizes = [int(size/2)]
            args.cfl_sizes = [size]
            args.sc_sizes = [size]
            
            args.use_bipartite_graphs = scenario in ["gnn","gnnff","gnnfe"]

            checkpoint_paths = {checkpoint:model_dir / f"epoch_{checkpoint:03d}.pt" for checkpoint in checkpoint_epochs}
            
            cache_key = (dataset, size, args.use_bipartite_graphs)
            
            M_fixed, N_fixed = infer_MN_from_ckpt(checkpoint_paths[1]) if scenario == "mlp" else (None, None)
            
            if cache_key not in dl_cache:
                train_loader, valid_loader, test_loader, N_max, M_max = build_dataloaders(
                    args, M_max=M_fixed, N_max=N_fixed, for_pretraining=False
                )
                dl_cache[cache_key] = (train_loader, valid_loader, test_loader, N_max, M_max)
            else:
                train_loader, valid_loader, test_loader, N_max, M_max = dl_cache[cache_key]
            
            model = (
            GNNPolicy(args).to(device)
                if args.use_bipartite_graphs
                else KKTNetMLP(args, M_fixed, N_fixed).to(device)
            )

            for checkpoint_epoch in tqdm(checkpoint_epochs):
                try:
                    print(f"Loading checkpoint {checkpoint_paths[checkpoint_epoch]}...")
                    args.encoder_path = str(checkpoint_paths[checkpoint_epoch])
                    
                    pkg = torch.load(checkpoint_paths[checkpoint_epoch], map_location="cpu")
                    if isinstance(pkg, dict):
                        if "model" in pkg:
                            model.load_state_dict(pkg["model"], strict=True)
                        elif "state_dict" in pkg:
                            model.load_state_dict(pkg["state_dict"], strict=True)
                    else:
                    # last resort
                        model.load_state_dict(pkg, strict=True)
                    
                    model = model.to(device)         
                    metrics = eval_epoch(model,test_loader, device, 0.1,0.1,0.6,0.2)

                    rows.append({
                                "size": size,
                                "dataset": dataset,
                                "scenario": scenario,
                                "run_dir": model_dir.name,
                                "epoch": checkpoint_epoch,
                                "ckpt_path": str(checkpoint_paths[checkpoint_epoch]),
                                **metrics,
                            })
                except Exception as e:
                    print(f"An exception occured for run: size:{size}, dataset: {dataset}, scenario:{scenario}, checkpoint: {checkpoint_epoch} | {e}")
                    continue



            


  0%|          | 0/3 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

M:28 N:10
args Namespace(encoder_path='/home/joachim-verschelde/Repos/KKT_MPNN/src/experiments/kkt_gnn_node_pretraining_experiments/run_gnn_policy-gv=2-lv=8-slice=512-point=13-lmbd=0.256-std=0.0-dim=128-l_mask=0.6-g_mask=0.05-l_e_mask=0.05-g_e_mask=0.6-dp=0.05-l_dim=128-conv=gatv2-heads=4_20260117_222826/best_encoder.pt', devices='0', batch_size=80, num_workers=0, seed=42, finetune_mode='full', use_bipartite_graphs=False, problems=['CA'], is_sizes=[10], ca_sizes=[5], sc_sizes=[10], cfl_sizes=[10], rnd_sizes=[200], n_instances=50000, data_root='../data/instances', val_split=0.15, lejepa_n_global_views=2, lejepa_n_local_views=8, sigreg_slices=512, sigreg_points=13, lejepa_lambda=0.256, lejepa_std_loss_weight=0.0, embedding_size=128, cons_nfeats=4, edge_nfeats=1, var_nfeats=18, num_emb_type='periodic', num_emb_bins=32, num_emb_freqs=16, lejepa_local_mask=0.6, lejepa_global_mask=0.05, lejepa_local_edge_mask=0.05, lejepa_global_edge_mask=0.6, dropout=0.05, lejepa_embed_dim=128, bipartite_co

  0%|          | 0/7 [00:00<?, ?it/s]

...

Loading checkpoint /home/joachim-verschelde/Repos/KKT_MPNN/src/experiments/kkt_gnn_node_finetuning_experiments/200/53-swift-darkness-all-gnn/epoch_001.pt...
Loading checkpoint /home/joachim-verschelde/Repos/KKT_MPNN/src/experiments/kkt_gnn_node_finetuning_experiments/200/53-swift-darkness-all-gnn/epoch_002.pt...
Loading checkpoint /home/joachim-verschelde/Repos/KKT_MPNN/src/experiments/kkt_gnn_node_finetuning_experiments/200/53-swift-darkness-all-gnn/epoch_005.pt...
Loading checkpoint /home/joachim-verschelde/Repos/KKT_MPNN/src/experiments/kkt_gnn_node_finetuning_experiments/200/53-swift-darkness-all-gnn/epoch_010.pt...
Loading checkpoint /home/joachim-verschelde/Repos/KKT_MPNN/src/experiments/kkt_gnn_node_finetuning_experiments/200/53-swift-darkness-all-gnn/epoch_020.pt...
Loading checkpoint /home/joachim-verschelde/Repos/KKT_MPNN/src/experiments/kkt_gnn_node_finetuning_experiments/200/53-swift-darkness-all-gnn/epoch_030.pt...
Loading checkpoint /home/joachim-verschelde/Repos/KKT_MPNN

  0%|          | 0/7 [00:00<?, ?it/s]

Loading checkpoint /home/joachim-verschelde/Repos/KKT_MPNN/src/experiments/kkt_gnn_node_finetuning_experiments/200/56-blooming-wind-all-gnnff/epoch_001.pt...
Loading checkpoint /home/joachim-verschelde/Repos/KKT_MPNN/src/experiments/kkt_gnn_node_finetuning_experiments/200/56-blooming-wind-all-gnnff/epoch_002.pt...
Loading checkpoint /home/joachim-verschelde/Repos/KKT_MPNN/src/experiments/kkt_gnn_node_finetuning_experiments/200/56-blooming-wind-all-gnnff/epoch_005.pt...
Loading checkpoint /home/joachim-verschelde/Repos/KKT_MPNN/src/experiments/kkt_gnn_node_finetuning_experiments/200/56-blooming-wind-all-gnnff/epoch_010.pt...
Loading checkpoint /home/joachim-verschelde/Repos/KKT_MPNN/src/experiments/kkt_gnn_node_finetuning_experiments/200/56-blooming-wind-all-gnnff/epoch_020.pt...
Loading checkpoint /home/joachim-verschelde/Repos/KKT_MPNN/src/experiments/kkt_gnn_node_finetuning_experiments/200/56-blooming-wind-all-gnnff/epoch_030.pt...
Loading checkpoint /home/joachim-verschelde/Repos/KK

  0%|          | 0/7 [00:00<?, ?it/s]

Loading checkpoint /home/joachim-verschelde/Repos/KKT_MPNN/src/experiments/kkt_gnn_node_finetuning_experiments/200/69-breezy-sky-all-gnnfe/epoch_001.pt...
Loading checkpoint /home/joachim-verschelde/Repos/KKT_MPNN/src/experiments/kkt_gnn_node_finetuning_experiments/200/69-breezy-sky-all-gnnfe/epoch_002.pt...
Loading checkpoint /home/joachim-verschelde/Repos/KKT_MPNN/src/experiments/kkt_gnn_node_finetuning_experiments/200/69-breezy-sky-all-gnnfe/epoch_005.pt...
Loading checkpoint /home/joachim-verschelde/Repos/KKT_MPNN/src/experiments/kkt_gnn_node_finetuning_experiments/200/69-breezy-sky-all-gnnfe/epoch_010.pt...
Loading checkpoint /home/joachim-verschelde/Repos/KKT_MPNN/src/experiments/kkt_gnn_node_finetuning_experiments/200/69-breezy-sky-all-gnnfe/epoch_020.pt...
Loading checkpoint /home/joachim-verschelde/Repos/KKT_MPNN/src/experiments/kkt_gnn_node_finetuning_experiments/200/69-breezy-sky-all-gnnfe/epoch_030.pt...
Loading checkpoint /home/joachim-verschelde/Repos/KKT_MPNN/src/experim

In [8]:
import pandas as pd
rows_df = pd.DataFrame(rows)

In [9]:
rows_df.head()

,size,dataset,scenario,run_dir,epoch,ckpt_path,valid/kkt_loss,valid/kkt_loss_std,valid/primal_feasibility,valid/primal_feasibility_std,valid/dual_feasibility,valid/dual_feasibility_std,valid/stationarity,valid/stationarity_std,valid/complementary_slackness,valid/complementary_slackness_std,valid/optimality_gap,valid/optimality_gap_std,valid/objective_gap,valid/objective_gap_std
0,10,ca,mlp,42-restful-resonance-ca-mlp,1,/home/joachim-verschelde/Repos/KKT_MPNN/src/ex...,2099.091566,1162.857241,0.000838,0.000873,0.0,0.0,2080.453172,1158.106984,18.637575,12.980373,0.127674,0.081711,0.147464,0.135410
1,10,ca,mlp,42-restful-resonance-ca-mlp,2,/home/joachim-verschelde/Repos/KKT_MPNN/src/ex...,1942.629845,1070.376810,0.002555,0.002149,0.0,0.0,1912.305301,1063.093617,30.321997,28.390083,0.102574,0.077661,0.076655,0.110446
2,10,ca,mlp,42-restful-resonance-ca-mlp,5,/home/joachim-verschelde/Repos/KKT_MPNN/src/ex...,1665.894185,987.584215,0.000673,0.000993,0.0,0.0,1634.112340,984.508728,31.781164,23.854945,0.356240,0.168503,0.195226,0.133197
3,10,ca,mlp,42-restful-resonance-ca-mlp,10,/home/joachim-verschelde/Repos/KKT_MPNN/src/ex...,551.390405,397.434804,0.006625,0.008053,0.0,0.0,486.168491,399.288357,65.215282,41.800714,0.290934,0.129168,0.023388,0.058252
4,10,ca,mlp,42-restful-resonance-ca-mlp,20,/home/joachim-verschelde/Repos/KKT_MPNN/src/ex...,335.612150,297.041765,0.005060,0.007353,0.0,0.0,283.398391,297.722533,52.208695,34.714217,0.274983,0.120632,0.045399,0.084166


In [10]:
rows_df.to_csv(out_csv, index=False)

In [11]:
rows_df.head()

,size,dataset,scenario,run_dir,epoch,ckpt_path,valid/kkt_loss,valid/kkt_loss_std,valid/primal_feasibility,valid/primal_feasibility_std,valid/dual_feasibility,valid/dual_feasibility_std,valid/stationarity,valid/stationarity_std,valid/complementary_slackness,valid/complementary_slackness_std,valid/optimality_gap,valid/optimality_gap_std,valid/objective_gap,valid/objective_gap_std
0,10,ca,mlp,42-restful-resonance-ca-mlp,1,/home/joachim-verschelde/Repos/KKT_MPNN/src/ex...,2099.091566,1162.857241,0.000838,0.000873,0.0,0.0,2080.453172,1158.106984,18.637575,12.980373,0.127674,0.081711,0.147464,0.135410
1,10,ca,mlp,42-restful-resonance-ca-mlp,2,/home/joachim-verschelde/Repos/KKT_MPNN/src/ex...,1942.629845,1070.376810,0.002555,0.002149,0.0,0.0,1912.305301,1063.093617,30.321997,28.390083,0.102574,0.077661,0.076655,0.110446
2,10,ca,mlp,42-restful-resonance-ca-mlp,5,/home/joachim-verschelde/Repos/KKT_MPNN/src/ex...,1665.894185,987.584215,0.000673,0.000993,0.0,0.0,1634.112340,984.508728,31.781164,23.854945,0.356240,0.168503,0.195226,0.133197
3,10,ca,mlp,42-restful-resonance-ca-mlp,10,/home/joachim-verschelde/Repos/KKT_MPNN/src/ex...,551.390405,397.434804,0.006625,0.008053,0.0,0.0,486.168491,399.288357,65.215282,41.800714,0.290934,0.129168,0.023388,0.058252
4,10,ca,mlp,42-restful-resonance-ca-mlp,20,/home/joachim-verschelde/Repos/KKT_MPNN/src/ex...,335.612150,297.041765,0.005060,0.007353,0.0,0.0,283.398391,297.722533,52.208695,34.714217,0.274983,0.120632,0.045399,0.084166


In [12]:
rows_ca_df = rows_df[rows_df.dataset=='ca']

In [13]:
rows_ca_df.columns

Index(['size', 'dataset', 'scenario', 'run_dir', 'epoch', 'ckpt_path',
       'valid/kkt_loss', 'valid/kkt_loss_std', 'valid/primal_feasibility',
       'valid/primal_feasibility_std', 'valid/dual_feasibility',
       'valid/dual_feasibility_std', 'valid/stationarity',
       'valid/stationarity_std', 'valid/complementary_slackness',
       'valid/complementary_slackness_std', 'valid/optimality_gap',
       'valid/optimality_gap_std', 'valid/objective_gap',
       'valid/objective_gap_std'],
      dtype='object')

In [30]:
import numpy as np
import pandas as pd

# Epoch budgets used in the paper tables
EPOCHS = [1, 2, 5, 10, 20, 30, 50]

# Row order used in the LaTeX tables
METHOD_ORDER = ["MLP-BL", "GNN-BL", "GNN-FE", "GNN-FF"]

# --- IMPORTANT ---
# Map your df["scenario"] values to the paper method names.
# Adjust this mapping to match your actual scenario strings.
#
# Tip: run `df["scenario"].unique()` once to see what you should map from.
SCENARIO_TO_METHOD = {
    "mlp": "MLP-BL",
    "gnn": "GNN-BL",
    "gnnfe": "GNN-FE",
    "gnnff": "GNN-FF",
    # If your scenarios include extra suffixes/prefixes, add them here, e.g.:
    # "mlp_bl_kkt": "MLP-BL",
    # "gnn_bl_kkt": "GNN-BL",
}


def budget_sweep_pivot_with_std(
    df: pd.DataFrame,
    size: int,
    *,
    dataset: str | None = None,
    metric: str = "valid/optimality_gap",
    metric_std: str | None = None,   # optional: if you already logged per-run std
    multiply_by_100: bool = True,
    std_ddof: int = 0,               # 0=population std across runs, 1=sample std
    cell_fmt: str = "{:.2f}",        # formatting for mean/std numbers
    pm_latex: bool = True,           # True -> "\pm", False -> "±"
) -> pd.DataFrame:
    """
    Returns a (Method x Epoch) table of strings "mean \\pm std" aggregated across runs.

    Aggregation:
      1) collapse duplicates within a run (same run_dir/epoch) by mean
      2) compute across-run mean/std for each (Method, epoch)
    """
    d = df.copy()

    # filter
    d = d[d["epoch"].isin(EPOCHS)]
    d = d[d["size"] == size]
    if dataset is not None:
        d = d[d["dataset"] == dataset]

    # map scenario -> Method
    d["Method"] = d["scenario"].map(SCENARIO_TO_METHOD).fillna(d["scenario"])

    if metric not in d.columns:
        raise KeyError(
            f"Metric column '{metric}' not found. Available columns: {list(d.columns)}"
        )

    # If you logged instance-level std per epoch as e.g. "valid/optimality_gap_std",
    # you can pass metric_std to display it. Otherwise we compute std across runs.
    has_logged_std = metric_std is not None and metric_std in d.columns

    # 1) de-duplicate within run/epoch (if needed)
    #    If std is logged, de-dup it too (mean of stds is usually fine for duplicates).
    group_cols = ["size", "dataset", "Method", "run_dir", "epoch"]
    agg_dict = {metric: "mean"}
    if has_logged_std:
        agg_dict[metric_std] = "mean"

    d1 = d.groupby(group_cols, as_index=False).agg(agg_dict)

    if multiply_by_100:
        d1[metric] = 100.0 * d1[metric]
        if has_logged_std:
            d1[metric_std] = 100.0 * d1[metric_std]

    # 2) aggregate across runs -> mean/std
    gcols2 = ["size", "dataset", "Method", "epoch"]

    if has_logged_std:
        # mean across runs for mean metric
        d2 = d1.groupby(gcols2, as_index=False)[metric].mean().rename(columns={metric: "mean"})
        # and (optionally) mean across runs of the logged instance-level std
        d2_std = d1.groupby(gcols2, as_index=False)[metric_std].mean().rename(columns={metric_std: "std"})
        d2 = d2.merge(d2_std, on=gcols2, how="left")
    else:
        d2 = (
            d1.groupby(gcols2, as_index=False)[metric]
              .agg(mean="mean", std=lambda x: float(np.std(x, ddof=std_ddof)))
        )

    pm = r"\pm" if pm_latex else "±"

    def format_cell(mu: float, sd: float) -> str:
        if pd.isna(mu):
            return np.nan
        sd = 0.0 if pd.isna(sd) else float(sd)
        return f"{cell_fmt.format(float(mu))} {pm} {cell_fmt.format(sd)}"

    d2["cell"] = [format_cell(mu, sd) for mu, sd in zip(d2["mean"], d2["std"])]

    # pivot to Method x Epoch (string table)
    pivot = d2.pivot_table(index="Method", columns="epoch", values="cell", aggfunc="first")

    # enforce order
    pivot = pivot.reindex(METHOD_ORDER)
    pivot = pivot.reindex(columns=EPOCHS)

    return pivot
def pivot_to_latex_table_booktabs(
    pivot: pd.DataFrame,
    *,
    caption: str,
    label: str,
    epochs: list[int],
    missing_cell: str = r"--",
    span_two_columns: bool = True,      # <---
    scale: str | None = r"\textwidth",  # r"\textwidth", r"\columnwidth", or None
    font_cmd: str = r"\small",          # r"\small", r"\footnotesize", r"\scriptsize"
    tabcolsep_pt: int | None = 3,       # set to None to not change
) -> str:
    header = ["Method"] + ["$E{=}1$" if e == 1 else str(e) for e in epochs]

    env = "table*" if span_two_columns else "table"
    width = scale  # choose \textwidth for table*, \columnwidth for table

    lines = []
    lines.append(fr"\begin{{{env}}}[t]")
    lines.append(r"\caption{" + caption + r"}")
    lines.append(r"\label{" + label + r"}")
    lines.append(r"\centering")
    lines.append(font_cmd)
    if tabcolsep_pt is not None:
        lines.append(rf"\setlength{{\tabcolsep}}{{{tabcolsep_pt}pt}}")

    if width is not None:
        lines.append(rf"\resizebox{{{width}}}{{!}}{{%")

    lines.append(r"\begin{tabular}{l" + "c"*len(epochs) + "}")
    lines.append(r"\toprule")
    lines.append(" & ".join(header) + r" \\")
    lines.append(r"\midrule")

    for method in METHOD_ORDER:
        row = [method]
        for e in epochs:
            x = pivot.loc[method, e] if (method in pivot.index and e in pivot.columns) else np.nan
            if pd.isna(x):
                row.append(missing_cell)
            else:
                # x is expected to be "mean \pm std"
                if isinstance(x, str) and r"\pm" in x:
                    mu_str, sd_str = x.split(r"\pm")
                    mu = float(mu_str.strip())
                    sd = float(sd_str.strip())
                    row.append(rf"\({mu:.1f} \pm {sd:.1f}\)")
                else:
                    # fallback (shouldn't usually happen)
                    row.append(rf"\({float(x):.1f}\)")
        lines.append(" & ".join(row) + r" \\")

    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")

    if width is not None:
        lines.append(r"}")  # end resizebox

    lines.append(fr"\end{{{env}}}")
    return "\n".join(lines)


# -------------------------
# USAGE
# -------------------------

# If you have multiple datasets in df, set this to the one you want (e.g. "combinatorial_auction").
# Otherwise leave as None.
DATASET_FILTER = None  # e.g. "combinatorial_auction"

# Build the three pivots
t10  = budget_sweep_pivot_with_std(rows_ca_df, 10,  dataset=DATASET_FILTER, metric="valid/optimality_gap", metric_std="valid/optimality_gap_std")
t50  = budget_sweep_pivot_with_std(rows_ca_df, 50,  dataset=DATASET_FILTER, metric="valid/optimality_gap", metric_std="valid/optimality_gap_std")
t200 = budget_sweep_pivot_with_std(rows_ca_df, 200, dataset=DATASET_FILTER, metric="valid/optimality_gap", metric_std="valid/optimality_gap_std")

# (Optional) Inspect as pandas tables
display(t10)
display(t50)
display(t200)

# Emit LaTeX matching your paper tables
latex10 = pivot_to_latex_table_booktabs(
    t10,
    caption=r"Validation relative optimality gap (\%) in the "
        r"Combinatorial Auction problem after $E$ fine-tuning epochs "
        r"for $n=10$. Lower is better.",
    label="tab:budget_sweep_n10",
    epochs=EPOCHS,
    span_two_columns=False,
    scale=r"\columnwidth",   # key bit
    font_cmd=r"\footnotesize",
    tabcolsep_pt=3,
)
latex50 = pivot_to_latex_table_booktabs(
    t50,
    caption=r"Validation relative optimality gap (\%) in the "
        r"Combinatorial Auction problem after $E$ fine-tuning epochs "
        r"for $n=50$. Lower is better.",
    label="tab:budget_sweep_n50",
    epochs=EPOCHS,
    span_two_columns=False,
    scale=r"\columnwidth",   # key bit
    font_cmd=r"\footnotesize",
    tabcolsep_pt=3,
)
latex200 = pivot_to_latex_table_booktabs(
    t200,
    caption=r"Validation relative optimality gap (\%) in the "
        r"Combinatorial Auction problem after $E$ fine-tuning epochs "
        r"for $n=200$. Lower is better.",
    label="tab:budget_sweep_n200",
    epochs=EPOCHS,
    span_two_columns=False,
    scale=r"\columnwidth",   # key bit
    font_cmd=r"\footnotesize",
    tabcolsep_pt=3,
)

print(latex10)
print()
print(latex50)
print()
print(latex200)


epoch,1,2,5,10,20,30,50
Method,,,,,,,
MLP-BL,12.77 \pm 8.17,10.26 \pm 7.77,35.62 \pm 16.85,29.09 \pm 12.92,27.50 \pm 12.06,28.75 \pm 11.85,22.45 \pm 10.84
GNN-BL,22.21 \pm 16.43,20.15 \pm 15.84,18.15 \pm 14.30,17.32 \pm 13.23,18.25 \pm 14.07,16.83 \pm 13.09,16.90 \pm 13.45
GNN-FE,23.97 \pm 17.91,20.27 \pm 15.87,19.55 \pm 15.43,19.18 \pm 15.23,19.00 \pm 14.94,18.95 \pm 15.12,18.96 \pm 15.14
GNN-FF,19.60 \pm 14.88,19.54 \pm 14.78,17.93 \pm 14.54,17.48 \pm 14.08,17.13 \pm 13.45,16.68 \pm 13.19,17.02 \pm 13.28


epoch,1,2,5,10,20,30,50
Method,,,,,,,
MLP-BL,36.22 \pm 18.60,10.23 \pm 8.46,31.18 \pm 9.09,17.42 \pm 6.21,12.39 \pm 6.51,13.07 \pm 5.75,11.08 \pm 5.22
GNN-BL,31.30 \pm 17.89,32.01 \pm 17.55,20.82 \pm 13.99,22.00 \pm 13.73,11.69 \pm 8.40,9.09 \pm 6.98,9.30 \pm 7.00
GNN-FE,33.45 \pm 19.39,15.02 \pm 11.19,24.81 \pm 13.29,20.80 \pm 12.58,17.26 \pm 11.48,15.91 \pm 10.95,18.57 \pm 11.64
GNN-FF,15.49 \pm 11.72,11.90 \pm 9.37,11.20 \pm 8.63,21.32 \pm 10.52,11.33 \pm 8.14,9.63 \pm 7.27,9.92 \pm 7.35


epoch,1,2,5,10,20,30,50
Method,,,,,,,
MLP-BL,45.04 \pm 26.71,19.07 \pm 3.61,15.47 \pm 6.69,21.69 \pm 7.60,19.77 \pm 8.63,23.36 \pm 8.91,21.62 \pm 8.94
GNN-BL,19.12 \pm 9.85,19.27 \pm 10.37,25.91 \pm 11.03,18.01 \pm 9.40,17.36 \pm 6.46,7.73 \pm 4.93,7.08 \pm 4.61
GNN-FE,21.69 \pm 9.77,25.32 \pm 7.70,20.61 \pm 7.94,26.26 \pm 7.66,19.71 \pm 7.41,24.13 \pm 7.45,22.44 \pm 7.32
GNN-FF,19.26 \pm 8.68,19.41 \pm 8.49,11.81 \pm 6.55,11.76 \pm 6.64,14.09 \pm 6.33,18.16 \pm 6.61,16.95 \pm 6.70


\begin{table}[t]
\caption{Validation relative optimality gap (\%) in the Combinatorial Auction problem after $E$ fine-tuning epochs for $n=10$. Lower is better.}
\label{tab:budget_sweep_n10}
\centering
\footnotesize
\setlength{\tabcolsep}{3pt}
\resizebox{\columnwidth}{!}{%
\begin{tabular}{lccccccc}
\toprule
Method & $E{=}1$ & 2 & 5 & 10 & 20 & 30 & 50 \\
\midrule
MLP-BL & \(12.8 \pm 8.2\) & \(10.3 \pm 7.8\) & \(35.6 \pm 16.9\) & \(29.1 \pm 12.9\) & \(27.5 \pm 12.1\) & \(28.8 \pm 11.8\) & \(22.4 \pm 10.8\) \\
GNN-BL & \(22.2 \pm 16.4\) & \(20.1 \pm 15.8\) & \(18.1 \pm 14.3\) & \(17.3 \pm 13.2\) & \(18.2 \pm 14.1\) & \(16.8 \pm 13.1\) & \(16.9 \pm 13.4\) \\
GNN-FE & \(24.0 \pm 17.9\) & \(20.3 \pm 15.9\) & \(19.6 \pm 15.4\) & \(19.2 \pm 15.2\) & \(19.0 \pm 14.9\) & \(18.9 \pm 15.1\) & \(19.0 \pm 15.1\) \\
GNN-FF & \(19.6 \pm 14.9\) & \(19.5 \pm 14.8\) & \(17.9 \pm 14.5\) & \(17.5 \pm 14.1\) & \(17.1 \pm 13.4\) & \(16.7 \pm 13.2\) & \(17.0 \pm 13.3\) \\
\bottomrule
\end{tabular}
}
\end{tab

In [ ]:
from matplotlib import pyplot as plt

def _scale_for_percent(values: pd.Series, df_gap_unit: str):
    """
    Returns (scale_factor, y_label).
    df_gap_unit:
      - "auto": if 95th percentile <= 1.0 assume fraction -> multiply by 100
      - "fraction": assume values in [0,1] -> multiply by 100
      - "percent": assume already in % units -> multiply by 1
    """
    v = values.dropna()
    if v.empty:
        return 1.0, "Relative optimality gap (%)"

    if df_gap_unit == "fraction":
        return 100.0, "Relative optimality gap (%)"
    if df_gap_unit == "percent":
        return 1.0, "Relative optimality gap (%)"
    if df_gap_unit == "auto":
        q95 = v.quantile(0.95)
        # Heuristic: if values mostly <= 1, it's probably a fraction
        return (100.0, "Relative optimality gap (%)") if q95 <= 1.0 else (1.0, "Relative optimality gap (%)")

    raise ValueError("df_gap_unit must be one of: 'auto', 'fraction', 'percent'")


def summarize_gap_over_runs(
    df: pd.DataFrame,
    size: int,
    *,
    dataset: str | None = None,
    metric: str = "valid/optimality_gap",
    epoch_subset: list[int] | None = EPOCHS,
    df_gap_unit: str = "auto",
) -> tuple[pd.DataFrame, str]:
    """
    Produces a tidy summary with columns:
      Method, epoch, mean, std, n_runs
    Aggregation:
      1) collapse duplicates within a run_dir at same epoch by mean
      2) aggregate across run_dir by mean + std
    """
    d = df.copy()

    d = d[d["size"] == size]
    if dataset is not None:
        d = d[d["dataset"] == dataset]

    if epoch_subset is not None:
        d = d[d["epoch"].isin(epoch_subset)]

    if metric not in d.columns:
        raise KeyError(f"'{metric}' not in df.columns")
    # scenario -> Method
    d["Method"] = d["scenario"].map(SCENARIO_TO_METHOD).fillna(d["scenario"])

    # Determine scaling for percent display
    scale, y_label = _scale_for_percent(d[metric], df_gap_unit=df_gap_unit)

    # 1) de-dup within run_dir/epoch (in case you logged multiple times)
    per_run = (
        d.groupby(["Method", "run_dir", "epoch"], as_index=False)[metric]
         .mean()
    )
    per_run[metric] = per_run[metric] * scale

    print(per_run.head())
    # 2) across runs
    summary = (
        per_run.groupby(["Method", "epoch"], as_index=False)[metric]
               .agg(mean="mean", std="std", n_runs="count")
    )
    print(summary.head())
    # Enforce order + epochs
    summary["Method"] = pd.Categorical(summary["Method"], categories=["mlp", "gnn", "gnnfe", "gnnff"], ordered=True)
    print(summary.head())
    summary = summary.sort_values(["Method", "epoch"])
    

    return summary, y_label


def plot_gap_vs_epoch(
    df: pd.DataFrame,
    size: int,
    *,
    dataset: str | None = None,
    metric: str = "valid/optimality_gap",
    epoch_subset: list[int] | None = EPOCHS,   # set None to plot ALL epochs present
    df_gap_unit: str = "auto",                 # "auto" | "fraction" | "percent"
    show_band: bool = True,                    # mean ± std band
    savepath: str | None = None,
):
    summary, y_label = summarize_gap_over_runs(
        df,
        size,
        dataset=dataset,
        metric=metric,
        epoch_subset=epoch_subset,
        df_gap_unit=df_gap_unit,
    )

    fig, ax = plt.subplots(figsize=(6.2, 3.9))

    for method in ["mlp", "gnn", "gnnfe", "gnnff"]:
        s = summary[summary["Method"] == method]
        if s.empty:
            continue

        # Line first (default matplotlib colors)
        (line,) = ax.plot(s["epoch"], s["mean"], marker="o", label=method)

        # Shaded band: mean ± std (if std exists)
        if show_band and "std" in s.columns:
            lo = (s["mean"] - s["std"]).to_numpy()
            hi = (s["mean"] + s["std"]).to_numpy()
            x = s["epoch"].to_numpy()
            ax.fill_between(x, lo, hi, alpha=0.2, color=line.get_color())

    ax.set_title(f"Optimality gap vs fine-tuning epoch (n={size})")
    ax.set_xlabel("Fine-tuning epoch $E$")
    ax.set_ylabel(y_label)
    ax.grid(True, alpha=0.3)
    ax.legend(frameon=False)

    # Nice x-ticks if you’re using the sweep budgets
    if epoch_subset is not None:
        ax.set_xticks(epoch_subset)

    fig.tight_layout()

    if savepath is not None:
        fig.savefig(savepath, bbox_inches="tight")

    return fig, ax


# -----------------------
# MAKE THE 3 PLOTS
# -----------------------

DATASET_FILTER = None  # e.g. "combinatorial_auction" if you want to filter

for n in [10, 50, 200]:
    plot_gap_vs_epoch(
        rows_ca_df,
        n,
        dataset=DATASET_FILTER,
        metric="valid/optimality_gap",
        epoch_subset=EPOCHS,     # or None for all epochs
        df_gap_unit="auto",      # or force "fraction" / "percent"
        show_band=True,
        savepath=f"opt_gap_vs_epoch_n{n}.pdf",  # comment out if you don't want files
    )

plt.show()